In [2]:
import os
import re
import sys
import gcsfs
import torch
import gc

from catboost import CatBoostClassifier, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

#from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from tab_transformer_pytorch import TabTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import subprocess
from google.cloud import storage
from google.cloud import bigquery


from tqdm import tqdm
import numpy as np
import pandas as pd
from collections import defaultdict

from transformers import AutoTokenizer, AutoModel
from datasets import Dataset
from itertools import islice

from datasets import load_dataset
from torch.optim import Adam
import torch.nn as nn
import torch.nn.functional as F

client = storage.Client()
bucket = client.bucket('cdow')

/opt/conda/lib/python3.10/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [3]:
client = bigquery.Client()
project_id = "expanded-nebula-754"
dataset_id = "sandbox_crdow"
table_id = "leads_training_set_text_category"
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# Use a wildcard * to shard the output into multiple files
destination_uri = "gs://cdow/leads_data/quality_leads_set-*.parquet"

extract_job = client.extract_table(
    table_ref,
    destination_uri,
    location="US",  # Adjust if your dataset is in a different location
    job_config=bigquery.job.ExtractJobConfig(
        destination_format="PARQUET"
    )
)

# Wait for job to complete
extract_job.result()
print(f"Exported {table_ref} to {destination_uri} in Parquet format")

/opt/conda/lib/python3.10/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Exported expanded-nebula-754.sandbox_crdow.leads_training_set_text_category to gs://cdow/leads_data/quality_leads_set-*.parquet in Parquet format


In [4]:
import pyarrow.dataset as ds
import gcsfs
fs = gcsfs.GCSFileSystem()

# List all files in the bucket folder
all_files = fs.ls("cdow/leads_data")
print(all_files)

# Filter only the Parquet shards for quality_leads_set
parquet_files = [
    f"gs://{file}" for file in all_files
    if file.startswith("cdow/leads_data/quality_leads_set-") and file.endswith(".parquet")
]

# Load the filtered Parquet files into a dataset using the GCS filesystem
dataset = ds.dataset(parquet_files, format="parquet", filesystem=fs)

# Convert to a PyArrow Table and then to a Pandas DataFrame
table = dataset.to_table()
df = table.to_pandas()

/opt/conda/lib/python3.10/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


['cdow/leads_data/', 'cdow/leads_data/embeddings.parquet', 'cdow/leads_data/leads_embeddings.parquet', 'cdow/leads_data/leads_embeddings_set.parquet', 'cdow/leads_data/leads_scoring.csv', 'cdow/leads_data/leads_scoring.parquet', 'cdow/leads_data/leads_scoring_set.parquet', 'cdow/leads_data/quality_leads_set-000000000000.parquet', 'cdow/leads_data/quality_leads_set-000000000001.parquet', 'cdow/leads_data/quality_leads_set-000000000002.parquet']


In [5]:
def preprocess_url(url):
    if pd.isna(url) or url.strip() == "":
        return "__MISSING__"

    # Remove protocol
    url = re.sub(r'^https?:\/\/', ' ', url)

    # Replace separators with space
    url = re.sub(r'[\/\.\?\=\&\%\:\;\_\-]', ' ', url)

    # Remove common file extensions and tokens m Body content 14382 id 9a63 4b7a 88f4 activityid 245b 42f0 bfff medium email
    url = re.sub(r'\b(html|php|aspx|www|RedirectTo|jsp|json|xml|utm|medium|source|sfmc|Banner|Imagecontent|listid|subscriberid|JobSubscriberBatchID|Body|Full_string|Logo Image URL|txt|@Logo Image URL)\b', ' ', url, flags=re.IGNORECASE)

    # Remove long alphanumeric strings (≥8 chars) that contain digits
    url = re.sub(r'\b(?=\w{8,})(?=\w*\d)\w+\b', ' ', url)
    # Remove exactly 2-digit numbers
    url = re.sub(r'\b\d{2}\b', ' ', url)

    # Remove exactly 3-character alphanumeric strings with at least one digit and one letter
    url = re.sub(r'\b(?=[a-zA-Z0-9]{3}$)(?=[a-zA-Z]*\d)(?=\d*[a-zA-Z])[a-zA-Z0-9]{4}\b', ' ', url)
    url = re.sub(r'\b(?=[a-zA-Z0-9]{3}$)(?=[a-zA-Z]*\d)(?=\d*[a-zA-Z])[a-zA-Z0-9]{3}\b', ' ', url)
    url = re.sub(r'\b(?=[a-zA-Z0-9]{3}$)(?=[a-zA-Z]*\d)(?=\d*[a-zA-Z])[a-zA-Z0-9]{2}\b', ' ', url)

    # Collapse multiple spaces
    url = re.sub(r'\s+', ' ', url).strip()

    return url

def process_url_list(url_list):
    # Handle NaN or None
    if url_list is None or isinstance(url_list, float) and pd.isna(url_list):
        return ["__MISSING__"]

    # Convert numpy array to list if needed
    if isinstance(url_list, np.ndarray):
        url_list = url_list.tolist()

    # Handle empty list
    if not url_list:
        return ["__MISSING__"]

    # Process each string
    return [preprocess_url(url) if isinstance(url, str) and url.strip() != "" else "__MISSING__" for url in url_list]




df['processed_url'] = df['a_url'].apply(process_url_list)

In [6]:
def collect_unique_strings(df, cols):
    """Collect all unique strings from list-columns of a dataframe."""
    unique_strings = set()
    
    for col in cols:
        def normalize(x):
            if isinstance(x, (float, type(None))):
                return []
            elif isinstance(x, np.ndarray):
                return x.tolist()
            elif isinstance(x, list):
                return x
            elif isinstance(x, str):
                return [x]
            else:
                return [str(x)]

        df[col] = df[col].apply(normalize)

        for row in df[col]:
            unique_strings.update(row)
    
    return list(unique_strings)



# -----------------------------
# 3. Helper: group by prefix
# -----------------------------
def group_by_prefix(strings, n_chars=10):
    """Group strings by first n_chars characters."""
    groups = {}
    for s in strings:
        prefix = s[:n_chars] if len(s) >= n_chars else s
        groups.setdefault(prefix, []).append(s)
    return groups

# -----------------------------
# 4. Build vocab with optional clustering per prefix group
# -----------------------------
def build_token_vocab(strings, n_prefix_chars=10, cluster_per_group=True, max_clusters_per_group=5):
    """
    Build token vocab from strings:
    - Group by prefix (first n_prefix_chars characters)
    - Optionally cluster each prefix group to reduce similar long strings
    Returns:
    - vocab: dict string -> token_id
    - cluster_info: dict prefix -> (vectorizer, kmeans) for inference
    """
    vocab = {}
    cluster_info = {}
    token_id = 2  # reserve 0,1 for special tokens
    groups = group_by_prefix(strings, n_chars=n_prefix_chars)
    
    for prefix, group_strings in groups.items():
        if cluster_per_group and len(group_strings) > 1:
            # Cluster similar strings within the prefix group
            n_clusters = min(max_clusters_per_group, len(group_strings))
            vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3,5))
            X = vectorizer.fit_transform(group_strings)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            labels = kmeans.fit_predict(X)
            cluster_info[prefix] = (vectorizer, kmeans)
            
            for s, label in zip(group_strings, labels):
                vocab[s] = token_id + label
            token_id += n_clusters
        else:
            # No clustering, all strings in this prefix share a token
            for s in group_strings:
                vocab[s] = token_id
            token_id += 1
    
    # Add special tokens
    vocab["__empty__"] = 0
    vocab["__missing__"] = 1
    return vocab, cluster_info

# -----------------------------
# 5. Map list of strings to tokens
# -----------------------------
def list_to_tokens(lst, vocab, cluster_info=None, n_prefix_chars=10, max_clusters_per_group=5):
    """Map a list of strings to token IDs."""
    if lst is None:
        return [vocab["__missing__"]]
    if len(lst) == 0:
        return [vocab["__empty__"]]
    
    tokens = []
    for s in lst:
        if s in vocab:
            tokens.append(vocab[s])
        else:
            # Handle unseen string
            prefix = s[:n_prefix_chars] if len(s) >= n_prefix_chars else s
            if cluster_info and prefix in cluster_info:
                vectorizer, kmeans = cluster_info[prefix]
                try:
                    X = vectorizer.transform([s])
                    label = kmeans.predict(X)[0]
                    # Map to token based on label and cluster offset
                    cluster_token = min(vocab.values()) + label + 2  # estimate
                    tokens.append(cluster_token)
                except:
                    tokens.append(len(vocab) + 1)  # UNK token
            else:
                tokens.append(len(vocab) + 1)  # UNK token
    return tokens

# -----------------------------
# 8. Helper functions for inference
# -----------------------------
def string_to_token_inference(s, vocab, cluster_info=None, n_prefix_chars=10):
    """Map a single string to token ID for inference."""
    if s is None or s == "":
        return vocab["__missing__"]
    if s in vocab:
        return vocab[s]
    prefix = s[:n_prefix_chars] if len(s) >= n_prefix_chars else s
    if cluster_info and prefix in cluster_info:
        vectorizer, kmeans = cluster_info[prefix]
        try:
            X = vectorizer.transform([s])
            label = kmeans.predict(X)[0]
            return min(vocab.values()) + label + 2  # approximate
        except:
            return len(vocab) + 1  # UNK
    return len(vocab) + 1  # UNK

def list_to_tokens_inference(lst, vocab, cluster_info=None, n_prefix_chars=10):
    """Map a list of strings to token IDs for inference."""
    if lst is None:
        return [vocab["__missing__"]]
    if len(lst) == 0:
        return [vocab["__empty__"]]
    return [string_to_token_inference(s, vocab, cluster_info, n_prefix_chars) for s in lst]



In [7]:
# -----------------------------
# 5. Map list of strings to tokens
# -----------------------------

all_email_strings = collect_unique_strings(df, ["a_emailsubject"])
all_url_strings = collect_unique_strings(df, ["processed_url"])
all_product_strings  =   collect_unique_strings(df,["a_product_interest"])  

#email_vocab, email_cluster_info = build_token_vocab(all_email_strings, n_prefix_chars=10, cluster_per_group=True)
#url_vocab, url_cluster_info = build_token_vocab(all_url_strings, n_prefix_chars=10, cluster_per_group=True)





In [8]:
# -----------------------------
# 6. Build vocab for emails and urls
# -----------------------------

email_vocab, email_cluster_info = build_token_vocab(all_email_strings, n_prefix_chars=10, cluster_per_group=True)
url_vocab, url_cluster_info = build_token_vocab(all_url_strings, n_prefix_chars=10, cluster_per_group=True)
product_vocab, product_cluster_info = build_token_vocab(all_product_strings, n_prefix_chars=10, cluster_per_group=True)



/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [8]:
print(product_cluster_info)

{'Advanced C': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=3, n_init=10, random_state=42)), 'Immunoassa': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=5, n_init=10, random_state=42)), 'Sample Pre': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=5, n_init=10, random_state=42)), 'Critical R': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=2, n_init=10, random_state=42)), 'Molecular ': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=5, n_init=10, random_state=42)), 'Other (Spe': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=3, n_init=10, random_state=42)), 'Environmen': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=3, n_init=10, random_state=42)), 'HPLC Colum': (TfidfVectorizer(analyzer='char', ngram_range=(3, 5)), KMeans(n_clusters=3, n_init=10, random_state=42)), 'In Process': (TfidfVectorizer(analyzer

In [9]:
# -----------------------------
# 7. Map columns to token lists
# -----------------------------
df["email_token_lists"] = df["a_emailsubject"].apply(
    lambda x: list_to_tokens(x, email_vocab, cluster_info=email_cluster_info, n_prefix_chars=10))
df["url_token_lists"] = df["processed_url"].apply(
    lambda x: list_to_tokens(x, url_vocab, cluster_info=url_cluster_info, n_prefix_chars=10))
df["product_token_lists"] = df["a_product_interest"].apply(
    lambda x: list_to_tokens(x, product_vocab, cluster_info=product_cluster_info, n_prefix_chars=10))#product_vocab, product_cluster_info 

In [10]:
print(df)
print("Email vocab size:", len(email_vocab))
print("URL vocab size:", len(url_vocab))
print("product vocab size:", len(product_vocab))

            opportunity_id                               a_email  \
0       006Hn00001Pzs68IAB                       [stefan@fot.bg]   
1       0061E00001KTT2UQAX  [thomas.paniucki@spectrumhealth.org]   
2       0061E00001JiH0cQAF        [glen.rowbothan@actrol.com.au]   
3       006dk000005cnPtAAI      [maintenance@premiermedcorp.com]   
4       006Hn00001QEJIhIAP                [mtla@novonordisk.com]   
...                    ...                                   ...   
204982  006Hn00001QFyGBIA1         [angelique.shaw@grainger.com]   
204983  0061E00001LH9B3QAL                 [oilay@rakmat.com.sg]   
204984  006Hn00001LqOwlIAF        [fisherparts@thermofisher.com]   
204985  006dk000002CMmnAAG           [daniele.lima@hc.fm.usp.br]   
204986  006dk000006lbyMAAQ          [amelia.james@perthmint.com]   

       a_search_terms a_source_campaigns  \
0                  []                 []   
1                  []                 []   
2                  []                 []   
3      

In [ ]:



# -----------------------------
# 7. Map columns to token lists
# -----------------------------
df["email_token_lists"] = df["email"].apply(
    lambda x: list_to_tokens(x, email_vocab, cluster_info=email_cluster_info, n_prefix_chars=10))
df["url_token_lists"] = df["url"].apply(
    lambda x: list_to_tokens(x, url_vocab, cluster_info=url_cluster_info, n_prefix_chars=10))

# -----------------------------
# 8. Helper functions for inference
# -----------------------------
def string_to_token_inference(s, vocab, cluster_info=None, n_prefix_chars=10):
    """Map a single string to token ID for inference."""
    if s is None or s == "":
        return vocab["__missing__"]
    if s in vocab:
        return vocab[s]
    prefix = s[:n_prefix_chars] if len(s) >= n_prefix_chars else s
    if cluster_info and prefix in cluster_info:
        vectorizer, kmeans = cluster_info[prefix]
        try:
            X = vectorizer.transform([s])
            label = kmeans.predict(X)[0]
            return min(vocab.values()) + label + 2  # approximate
        except:
            return len(vocab) + 1  # UNK
    return len(vocab) + 1  # UNK

def list_to_tokens_inference(lst, vocab, cluster_info=None, n_prefix_chars=10):
    """Map a list of strings to token IDs for inference."""
    if lst is None:
        return [vocab["__missing__"]]
    if len(lst) == 0:
        return [vocab["__empty__"]]
    return [string_to_token_inference(s, vocab, cluster_info, n_prefix_chars) for s in lst]

# -----------------------------
# 9. Inspect result
# -----------------------------
print(df)
print("Email vocab size:", len(email_vocab))
print("URL vocab size:", len(url_vocab))
